# 06 – Style versus framing

> **Updated 2026-10-29.** Scored on **full text** (the main H1 scoring) with lexicon v1.2 and the H1 gate open. The decision rules below were written before the first run of this notebook (on 300-word cuts, lexicon v1.1) and are unchanged for this run.

**The problem.** A classifier separates chatbot answers from news almost perfectly, and that proves nothing about framing: chatbot answers were asked for plain prose without headings, news has quotes, numbers, short paragraphs, datelines. Several lexicon terms (`ensure`, `regulation`, `approach`, `accountability`) are also typical chatbot vocabulary. So we need to show how much of the chatbot/news difference is **style** and how much is **frame mix**.

**Two classifiers**
- **A – style only.** Function-word rates, punctuation, sentence length and formatting markers. **Every lexicon term is removed from the features** (checked by an assertion).
- **B – frame mix only.** The six frame-hit proportions of a text, nothing else.

**Held-out evaluation** (a classifier is judged only on texts it has not seen):
1. **Outlets/questions held out** (5-fold grouped CV: all articles of an outlet, all answers to a question, stay together).
2. **Model held out:** train on one chatbot's answers (+ news from training outlets), test on the *other* chatbot's answers (+ news from held-out outlets). Run in both directions.

**Decision rules, written before looking at any result**

| Result | What it would indicate |
|---|---|
| A held-out AUC ≥ 0.90 | chatbot and news are strongly separable by style alone (expected) |
| B held-out AUC ≤ 0.60 | the frame mix alone barely identifies the source: frame differences are small next to style differences |
| B held-out AUC ≥ 0.65 **and** it stays ≥ 0.65 when a chatbot not seen in training is tested | the frame mix carries source information of its own, beyond style (framing difference, or at least not explained by style) |
| A + B AUC clearly above A alone (more than 0.02) | frame mix adds information that style does not already contain |
| B AUC between 0.60 and 0.65 | inconclusive; report it as such |

Because B uses only texts with at least 5 hits (all-zero vectors would let the classifier separate sources by hit density alone), A is also evaluated on that same subset so A and B are compared like for like.

**A style-matched subset** is also built (nearest-neighbour matching on sentence length, commas, paragraphs and digits) and its balance is reported. The H1 entropy comparison on that subset is written but gated behind `H_TESTS_ENABLED` like the rest of H1 (see `05_rarefied_entropy.ipynb`).

AUC is a measure of separability, not a hypothesis test of H1–H3.

In [1]:
import json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from framescore import Lexicon, lem, MAX_WORDS

H_TESTS_ENABLED = True        # same gate as notebook 05; opened 2026-10-29 (lexicon v1.2 is signed off)
SEED = 42
CFG = json.load(open("config.json"))
OUT = Path(CFG["analysis_dir"]); OUT.mkdir(parents=True, exist_ok=True)
assert json.load(open(CFG["lexicon_json"]))["signed_off"], "the lexicon is not signed off"
LEX = Lexicon(CFG["lexicon_json"]); FRAMES = LEX.frames; HC = [f"hits_{f}" for f in FRAMES]
MODELS = {"Llama": "llama3.1:8b", "Qwen": "qwen3.5:9b"}

## 1. Texts, keeping line breaks
Main scoring is **full text** (decision 2026-10-29, see `dataset/analysis/DECISIONS.md`): chatbot answers whole, news articles whole. The frame scorer flattens whitespace; formatting markers (paragraph breaks, bullets) need the original line breaks, so the texts are used as written. Frame hits come from `frame_scores_*_full.csv` (notebook 03, lexicon v1.2).

In [2]:
ans = pd.read_json(CFG["answers_jsonl"], lines=True)
ans = ans[ans["chatbot"].isin(MODELS.values())].reset_index(drop=True)
chat_scores = pd.read_csv(OUT / "frame_scores_chatbot_full.csv")
assert (chat_scores["prompt_id"] == ans["prompt_id"]).all() and (chat_scores["replicate"] == ans["replicate"]).all()
chat = pd.DataFrame({"label": 1, "model": ans["chatbot"].map({v: k for k, v in MODELS.items()}), "group": "q_" + ans["question_id"],
                     "text": ans["answer"], "question_id": ans["question_id"]})
chat[HC] = chat_scores[HC].to_numpy(); chat["hits_total"] = chat[HC].sum(axis=1)

news_raw = pd.read_csv(CFG["news_corpus_csv"], keep_default_na=False)
news_scores = pd.read_csv(OUT / "frame_scores_news_full.csv")
assert (news_raw["doc_id"] == news_scores["doc_id"]).all()
news = pd.DataFrame({"label": 0, "model": "news", "group": "o_" + news_raw["outlet"], "text": news_raw["text"], "question_id": ""})
news[HC] = news_scores[HC].to_numpy(); news["hits_total"] = news[HC].sum(axis=1)
D = pd.concat([chat, news], ignore_index=True)
print(len(chat), "chatbot answers,", len(news), "news articles;", "paragraph breaks present in news text:", news["text"].str.contains("\n").mean().round(2),
      "| in chatbot answers:", chat["text"].str.contains("\n").mean().round(2))

216 chatbot answers, 420 news articles; paragraph breaks present in news text: 1.0 | in chatbot answers: 0.75


## 2. Features
**A (style):** function-word rates per 100 words + 14 style statistics. The function-word list is scikit-learn's English stop-word list **minus** any word that is a lexicon term (single-word terms, compared on lemmas) and a few content words that list contains by accident. **B:** the six frame-hit proportions.

In [3]:
lex_lemmas = {l for l in LEX.owner} | {l for l in LEX.jur_owner}
content_words = {"bill", "fire", "system", "amount", "cry", "mill", "interest", "serious", "front", "back", "empty", "full", "found", "call", "detail",
                 "describe", "fill", "find", "thick", "thin", "top", "bottom", "side", "part", "show", "move", "keep", "put", "give", "made", "make", "take", "see", "seem"}
function_words = sorted(w for w in ENGLISH_STOP_WORDS if lem(w) not in lex_lemmas and w not in lex_lemmas and w not in content_words)
removed = sorted(w for w in ENGLISH_STOP_WORDS if w not in function_words)
assert not any(lem(w) in lex_lemmas or w in lex_lemmas for w in function_words), "a lexicon term is still among the style features"
print(len(function_words), "function words kept;", len(removed), "removed from the stop list:", removed)

def style_features(t):
    words = re.findall(r"[A-Za-z']+", t); lw = [w.lower() for w in words]; n = max(len(words), 1)
    sents = [s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s]; sl = [len(re.findall(r"[A-Za-z']+", s)) for s in sents] or [0]
    per100 = lambda c: 100 * c / n
    f = {"sent_len_mean": np.mean(sl), "sent_len_sd": np.std(sl), "word_len_mean": np.mean([len(w) for w in words]) if words else 0,
         "commas_per100": per100(t.count(",")), "semicolons_per100": per100(t.count(";")), "colons_per100": per100(t.count(":")),
         "dashes_per100": per100(len(re.findall(r"[\u2013\u2014]|\s-\s", t))), "parens_per100": per100(t.count("(")),
         "quotes_per100": per100(len(re.findall(r"[\"\u201c\u201d]", t))), "question_excl_per100": per100(t.count("?") + t.count("!")),
         "digits_per100": per100(len(re.findall(r"\d+", t))), "capitalised_share": np.mean([w[0].isupper() for w in words]) if words else 0,
         "paragraphs_per100": per100(len([p for p in re.split(r"\n+", t.strip()) if p])), "bullets_per100": per100(len(re.findall(r"(?m)^\s*(?:[-*\u2022]|\d+[.)])\s", t)))}
    c = pd.Series(lw).value_counts()
    f.update({"fw_" + w: per100(c.get(w, 0)) for w in function_words})
    return f

SF = pd.DataFrame([style_features(t) for t in D["text"]])
style_cols = list(SF.columns); STYLE_STATS = [c for c in style_cols if not c.startswith("fw_")]
share = D[HC].div(D["hits_total"].replace(0, np.nan), axis=0).fillna(0).to_numpy()
B_cols = [f"share_{f}" for f in FRAMES]; D[B_cols] = share
D = pd.concat([D, SF], axis=1)
enough = D["hits_total"] >= 5
print("features A:", len(style_cols), "| B:", len(B_cols), "| texts with >= 5 hits (used for B):", int(enough.sum()), "of", len(D), "| chatbot:", int(enough[D.label == 1].sum()), "news:", int(enough[D.label == 0].sum()))

285 function words kept; 33 removed from the stop list: ['amount', 'back', 'bill', 'bottom', 'call', 'cry', 'describe', 'detail', 'empty', 'fill', 'find', 'fire', 'found', 'front', 'full', 'give', 'interest', 'keep', 'made', 'mill', 'move', 'part', 'put', 'see', 'seem', 'serious', 'show', 'side', 'system', 'take', 'thick', 'thin', 'top']


features A: 299 | B: 6 | texts with >= 5 hits (used for B): 629 of 636 | chatbot: 216 news: 413


## 3. Held-out evaluation

In [4]:
def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000, class_weight="balanced"))

def scores(y, p):
    return {"auc": roc_auc_score(y, p), "bal_acc": balanced_accuracy_score(y, p > 0.5)}

def grouped_cv(data, cols, n_splits=5):
    out = []
    for tr, te in GroupKFold(n_splits).split(data, data["label"], data["group"]):
        m = make_model().fit(data.iloc[tr][cols], data.iloc[tr]["label"])
        out.append(scores(data.iloc[te]["label"], m.predict_proba(data.iloc[te][cols])[:, 1]))
    return pd.DataFrame(out)

def model_held_out(data, cols, train_model, n_splits=5):
    """Train on one chatbot (+ news of training outlets); test on the other chatbot (+ news of held-out outlets)."""
    test_model = "Qwen" if train_model == "Llama" else "Llama"
    nws = data[data["label"] == 0].reset_index(drop=True); out = []
    for tr, te in GroupKFold(n_splits).split(nws, nws["label"], nws["group"]):
        train = pd.concat([data[data["model"] == train_model], nws.iloc[tr]]); test = pd.concat([data[data["model"] == test_model], nws.iloc[te]])
        m = make_model().fit(train[cols], train["label"])
        out.append(scores(test["label"], m.predict_proba(test[cols])[:, 1]))
    return pd.DataFrame(out)

sets = {"A style (all texts)": (D, style_cols), "A style (texts with >= 5 hits)": (D[enough].reset_index(drop=True), style_cols),
        "B frame mix (texts with >= 5 hits)": (D[enough].reset_index(drop=True), B_cols),
        "A + B (texts with >= 5 hits)": (D[enough].reset_index(drop=True), style_cols + B_cols),
        "B frame mix (all texts, zero-hit texts = all-zero vector)": (D, B_cols)}
rows = []
for name, (data, cols) in sets.items():
    for split, res in [("outlets/questions held out", grouped_cv(data, cols)),
                       ("train Llama, test Qwen", model_held_out(data, cols, "Llama")),
                       ("train Qwen, test Llama", model_held_out(data, cols, "Qwen"))]:
        rows.append({"features": name, "split": split, "n_folds": len(res), "auc_mean": res["auc"].mean(), "auc_sd": res["auc"].std(),
                     "balanced_acc_mean": res["bal_acc"].mean()})
clf = pd.DataFrame(rows).round(3)
clf.to_csv(OUT / "style_vs_framing_classifiers.csv", index=False)
clf

,features,split,n_folds,auc_mean,auc_sd,balanced_acc_mean
0,A style (all texts),outlets/questions held out,5,1.000,0.000,1.000
1,A style (all texts),"train Llama, test Qwen",5,0.998,0.000,0.888
2,A style (all texts),"train Qwen, test Llama",5,0.999,0.001,0.899
3,A style (texts with >= 5 hits),outlets/questions held out,5,1.000,0.000,0.999
4,A style (texts with >= 5 hits),"train Llama, test Qwen",5,0.998,0.002,0.887
5,A style (texts with >= 5 hits),"train Qwen, test Llama",5,0.999,0.001,0.901
6,B frame mix (texts with >= 5 hits),outlets/questions held out,5,0.700,0.071,0.647
7,B frame mix (texts with >= 5 hits),"train Llama, test Qwen",5,0.741,0.033,0.681
8,B frame mix (texts with >= 5 hits),"train Qwen, test Llama",5,0.719,0.011,0.663
9,A + B (texts with >= 5 hits),outlets/questions held out,5,1.000,0.000,0.999


In [5]:
# which style features carry the separation? (standardised logistic coefficients, fitted on everything; descriptive only)
m = make_model().fit(D[style_cols], D["label"]); coef = pd.Series(m[-1].coef_[0], index=style_cols)
top = pd.concat([coef.sort_values().head(8), coef.sort_values().tail(8)])
top.to_frame("coef (positive = chatbot-like)").round(2)

,coef (positive = chatbot-like)
capitalised_share,-0.40
paragraphs_per100,-0.35
sent_len_sd,-0.26
fw_has,-0.24
digits_per100,-0.24
quotes_per100,-0.24
fw_at,-0.17
parens_per100,-0.17
fw_that,0.18
word_len_mean,0.18


In [6]:
# frame-mix classifier B: which frames separate the sources? (mean share per source among texts with >= 5 hits)
D[enough].groupby("model")[B_cols].mean().round(3)

,share_innovation_opportunity,share_risk_safety,share_regulation_governance,share_rights_privacy,share_economy_labour,share_misinformation_integrity
model,,,,,,
Llama,0.211,0.221,0.261,0.099,0.127,0.081
Qwen,0.168,0.251,0.314,0.075,0.128,0.063
news,0.117,0.256,0.228,0.097,0.242,0.060


## 4. Style-matched subsets
Each chatbot answer is matched (nearest neighbour on standardised style features; no news chunk used twice; caliper 0.5 standard deviations per feature) to one news chunk. Two specifications: **2 features** (sentence length, commas per 100 words) and **4 features** (+ paragraphs per 100 words, digits per 100 words). The balance table reports the standardised mean difference (SMD) of all four features before and after matching; |SMD| below 0.1 is the usual bar for "balanced". A subset in which the style gap has gone is where an entropy difference could no longer be put down to style.

In [7]:
ALLF = ["sent_len_mean", "commas_per100", "paragraphs_per100", "digits_per100"]
SPECS = {"2 features": ["sent_len_mean", "commas_per100"], "4 features": ALLF}
SPEC_MAIN = "2 features"
mu, sd = D[ALLF].mean(), D[ALLF].std().replace(0, 1)

def match(model, feats, caliper=0.5, seed=SEED):
    a = D[D["model"] == model]; n = D[D["label"] == 0]
    Za, Zn = ((a[feats] - mu[feats]) / sd[feats]).to_numpy(), ((n[feats] - mu[feats]) / sd[feats]).to_numpy()
    used, pairs = set(), []
    for i in np.random.default_rng(seed).permutation(len(a)):
        d = np.sqrt(((Zn - Za[i]) ** 2).sum(axis=1)); d[list(used)] = np.inf
        j = int(d.argmin())
        if d[j] <= caliper * np.sqrt(len(feats)):
            used.add(j); pairs.append((a.index[i], n.index[j]))
    return pairs

pairs_all = {(spec, m): match(m, feats) for spec, feats in SPECS.items() for m in MODELS}
smd = lambda x, y: (x.mean() - y.mean()) / np.sqrt((x.var() + y.var()) / 2)
rows = []
for (spec, m), pr in pairs_all.items():
    a_all, n_all = D[D["model"] == m], D[D["label"] == 0]
    a_m, n_m = D.loc[[p[0] for p in pr]], D.loc[[p[1] for p in pr]]
    for f in ALLF:
        rows.append({"spec": spec, "chatbot": m, "feature": f, "matched_on": f in SPECS[spec], "SMD_before": smd(a_all[f], n_all[f]),
                     "SMD_after": smd(a_m[f], n_m[f]), "pairs_kept": len(pr), "chatbot_texts": len(a_all)})
balance = pd.DataFrame(rows).round(2); balance.to_csv(OUT / "style_matching_balance.csv", index=False)
pairs = {m: pairs_all[(SPEC_MAIN, m)] for m in MODELS}        # the pairs used for the (gated) entropy comparison
balance

,spec,chatbot,feature,matched_on,SMD_before,SMD_after,pairs_kept,chatbot_texts
0,2 features,Llama,sent_len_mean,True,-0.47,-0.01,107,108
1,2 features,Llama,commas_per100,True,-0.32,-0.02,107,108
2,2 features,Llama,paragraphs_per100,False,-1.88,-2.02,107,108
3,2 features,Llama,digits_per100,False,-1.31,-1.44,107,108
4,2 features,Qwen,sent_len_mean,True,0.28,0.03,108,108
5,2 features,Qwen,commas_per100,True,-0.04,0.01,108,108
6,2 features,Qwen,paragraphs_per100,False,-3.12,-3.38,108,108
7,2 features,Qwen,digits_per100,False,-1.31,-1.47,108,108
8,4 features,Llama,sent_len_mean,True,-0.47,-0.12,71,108
9,4 features,Llama,commas_per100,True,-0.32,-0.02,71,108


In [8]:
# Are the matched subsets still separable by style? (classifier A on matched pairs only, grouped CV)
rows = []
for (spec, m), pr in pairs_all.items():
    idx = [p[0] for p in pr] + [p[1] for p in pr]
    sub = D.loc[idx].reset_index(drop=True)
    if sub["group"].nunique() >= 5 and len(pr) >= 10:
        r = grouped_cv(sub, style_cols); rows.append({"spec": spec, "matched subset": m, "pairs": len(pr), "A style AUC (held-out groups)": r["auc"].mean().round(3)})
pd.DataFrame(rows)

,spec,matched subset,pairs,A style AUC (held-out groups)
0,2 features,Llama,107,1.0
1,2 features,Qwen,108,1.0
2,4 features,Llama,71,1.0
3,4 features,Qwen,24,1.0


## 5. Entropy comparison on the style-matched subsets (gated)
Rarefied entropy (K = 5, from `05_rarefied_entropy.ipynb`) on matched pairs where both texts have at least 5 hits. Paired difference, chatbot minus news, with a cluster bootstrap over questions. Written now; run on real groups only when the gate is open. The self-test below checks the code on shuffled entropy values.

In [9]:
rar = pd.read_csv(OUT / "rarefied_entropy_per_text.csv")
chat_rar = {m: rar[rar["view"] == f"{m} (full text)"].reset_index(drop=True) for m in MODELS}
news_rar = rar[rar["view"] == "news (full text)"].reset_index(drop=True)
chat_idx = {m: D[D["model"] == m].index for m in MODELS}

def matched_entropy_comparison(shuffle=False, n_boot=2000, seed=SEED):
    rows = []; r = np.random.default_rng(seed)
    for m, pr in pairs.items():
        a_pos = {ix: k for k, ix in enumerate(chat_idx[m])}; n_off = len(chat)
        ea = chat_rar[m]["rar5"].to_numpy(); en = news_rar["rar5"].to_numpy()
        if shuffle:
            allv = np.concatenate([ea, en]); r.shuffle(allv); ea, en = allv[:len(ea)], allv[len(ea):]
        d = pd.DataFrame([(ea[a_pos[a]], en[n - n_off], D.loc[a, "question_id"]) for a, n in pr], columns=["chatbot", "news", "q"]).dropna()
        d["diff"] = d["chatbot"] - d["news"]
        g = d.groupby("q")["diff"]; sums, cnt = g.sum().to_numpy(), g.count().to_numpy()
        idx = r.integers(0, len(sums), (n_boot, len(sums))); boot = sums[idx].sum(axis=1) / cnt[idx].sum(axis=1)
        rows.append({"chatbot": m, "pairs_with_both_scored": len(d), "diff_mean": d["diff"].mean(), "diff_lo": np.percentile(boot, 2.5), "diff_hi": np.percentile(boot, 97.5)})
    return pd.DataFrame(rows).round(4)

selftest = matched_entropy_comparison(shuffle=True)
assert len(selftest) == 2 and selftest["pairs_with_both_scored"].min() > 10
print("self-test on shuffled entropy values passed (numbers are noise and are not saved); mean |diff| =", selftest["diff_mean"].abs().mean().round(3))
if H_TESTS_ENABLED:
    res = matched_entropy_comparison(); res.to_csv(OUT / "style_matched_entropy_comparison.csv", index=False); display(res)
else:
    print("H_TESTS_ENABLED is False: the matched-subset entropy comparison on real groups is not computed.")

self-test on shuffled entropy values passed (numbers are noise and are not saved); mean |diff| = 0.016


,chatbot,pairs_with_both_scored,diff_mean,diff_lo,diff_hi
0,Llama,105,-0.0134,-0.0470,0.0195
1,Qwen,107,0.0570,0.0328,0.0771


## 6. Interpretation (against the decision rules in the header; full text, lexicon v1.2)

**What would indicate style, and what would indicate framing.**
- *Style:* classifier A (style only, every lexicon term removed) separates chatbot from news almost perfectly, classifier B (frame mix only) barely beats chance, and the H1 entropy gap disappears in a subset matched on style.
- *Framing:* B separates the sources clearly **on texts it has not seen** (held-out outlets and questions, and a chatbot model not used in training), A + B beats A, and the entropy gap survives style matching.

| Rule | Result | Verdict |
|---|---|---|
| A held-out AUC ≥ 0.90 | A = 1.000 (outlets/questions held out), 0.998 and 0.999 (model held out, both directions) | **met**: style alone separates chatbot from news essentially perfectly |
| B held-out AUC ≤ 0.60 | B = 0.700 (outlets/questions held out) | not met |
| B ≥ 0.65 and still ≥ 0.65 on a chatbot not seen in training | B = 0.741 (train Llama, test Qwen) and 0.719 (train Qwen, test Llama) | **met**: the frame mix carries source information beyond chance for unseen outlets and an unseen model |
| A + B clearly above A | A + B = 1.000, 0.998, 0.999, the same as A | **cannot be assessed**: A is at the ceiling |

**What this means.** (1) Style alone identifies the source, so any classifier on raw text would say nothing about framing. (2) The frame mix alone gives AUC 0.70–0.74, clearly above chance and far below 1.0, and it holds for a chatbot the classifier never saw: the frame mix is **not a pure style artefact**. (3) The difference is in *which* frames, not in evenness: among texts with at least 5 hits the mean frame share is economy/labour 0.24 in news against 0.13 in both chatbots, innovation 0.21 (Llama) and 0.17 (Qwen) against 0.12 in news, and governance 0.31 (Qwen) and 0.26 (Llama) against 0.23 in news. (4) AUC measures separability, not a difference in entropy, and several lexicon terms are partly register words, so this supports "the frame mix differs and is not just style" but does not by itself show a difference in *emphasis* in the sense of H1.

**Style-matched entropy comparison (K = 5, full text).** Matched on sentence length and commas (|SMD| after matching 0.01–0.03, 107–108 pairs per chatbot): Llama − news = −0.013 [−0.047, +0.020] (105 pairs scored), Qwen − news = +0.057 [+0.033, +0.077] (107 pairs). These match the unmatched main result (Llama −0.004, Qwen +0.058): matching those two features changes nothing. But **formatting cannot be matched**: paragraph breaks and digits stay far apart after matching (SMD −2.0 and −1.4 for Llama, −3.4 and −1.5 for Qwen), the 4-feature specification keeps only 71 Llama and 24 Qwen pairs, and classifier A still separates every matched subset perfectly (AUC 1.0). The matched comparison therefore removes only the sentence-length and comma differences; formatting differences remain and must be stated as a limitation.

## Limits to state in the report
- The function-word list is a generic English list, not tuned to this corpus; punctuation and formatting features exist partly because of how the two corpora were produced (answers were asked for plain prose without bullets). That is exactly why the style classifier succeeds: it measures production differences, not meaning.
- B depends on the lexicon; if the lexicon is biased towards chatbot vocabulary, B overstates framing differences. The ambiguous-term and prompt-echo robustness runs in notebook 03 address this at the scoring level.
- 18 question groups limit the grouped cross-validation on the chatbot side.
- Matching on four style variables cannot remove every style difference; the balance table says how much remains.